# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import base64
import numpy as np
import wave
import io


In [ ]:
# Constants I wont image model due due to costs.
CHAT_MODEL = "openai/gpt-5-mini"
#IMAGE_MODEL = "openai/gpt-5-image-mini" 
AUDIO_MODEL = "openai/gpt-audio-mini"

In [3]:
# set up environment
load_dotenv(override=True)

openai_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_url = os.getenv('OPENROUTER_API_URL')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    print(f"OpenAI API Url exists and is {openai_api_url}")
else:
    print("OpenAI API Key not set")
    
client = OpenAI(
    base_url=openai_api_url, 
    api_key=openai_api_key,
)




OpenAI API Key exists and begins sk-or-v1
OpenAI API Url exists and is https://openrouter.ai/api/v1


In [4]:
# Chat messages

system_message = """
You are a helpful technical expert for Q&A forum.
Give short, courteous answers, no more than 10 sentence.
Always be accurate. If you don't know the answer, say so.
"""

user_message = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
    ]

In [5]:
import base64
import numpy as np
import wave
import io

def talker(message):
    stream = client.chat.completions.create(
      model=AUDIO_MODEL,
      modalities=["text", "audio"],
      audio={"voice": "alloy", "format": "pcm16"},
      stream=True, 
      messages=[
          {
              "role": "user",
              "content": message  
          }
      ]
    )
     
    # 2. Iterate through the stream to collect audio chunks
    full_audio_base64 = ""
    for chunk in stream:
        if hasattr(chunk.choices[0].delta, 'audio') and chunk.choices[0].delta.audio:
            if 'data' in chunk.choices[0].delta.audio:
                # Append the raw base64 chunks
                full_audio_base64 += chunk.choices[0].delta.audio['data']

    if not full_audio_base64:
        return None

    # 2. Convert base64 to NumPy array (Gradio's preferred format)
    audio_bytes = base64.b64decode(full_audio_base64)
    
    # Padding fix for pcm16 byte alignment
    if len(audio_bytes) % 2 != 0:
        audio_bytes += b'\x00'
        
    audio_data = np.frombuffer(audio_bytes, dtype=np.int16)
    
    # 3. Return (Sample Rate, Data) to Gradio
    # OpenAI pcm16 is standard 24000Hz
    return (24000, audio_data)
    



In [ ]:
def process_audio(message, audio_data):
    if audio_data is None:
        return None
    
    # 1. Gradio mic input returns (sample_rate, numpy_array)
    sr, y = audio_data
    
    # 2. Convert to WAV bytes in-memory
    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as wav_file:
        wav_file.setnchannels(1) # Mono
        wav_file.setsampwidth(2) # 16-bit
        wav_file.setframerate(sr)
        wav_file.writeframes(y.tobytes())
    
    # 3. Encode to Base64 for the API
    base64_audio = base64.b64encode(buffer.getvalue()).decode("utf-8")


    stream = client.chat.completions.create(
      model=AUDIO_MODEL,
      modalities=["text", "audio"],
      audio={"voice": "alloy", "format": "pcm16"},
      stream=True,
      messages=[
            {
              "role": "user",
              "content": message  
            },
            {
                "role": "user",
                "content": [
                    {"type": "input_audio", "input_audio": {"data": base64_audio, "format": "wav"}}
                ]
            }
        ]
    )
     
    # 2. Iterate through the stream to collect audio chunks
    full_audio_base64 = ""
    for chunk in stream:
        if hasattr(chunk.choices[0].delta, 'audio') and chunk.choices[0].delta.audio:
            if 'data' in chunk.choices[0].delta.audio:
                # Append the raw base64 chunks
                full_audio_base64 += chunk.choices[0].delta.audio['data']

    if not full_audio_base64:
        return None

    # 2. Convert base64 to NumPy array (Gradio's preferred format)
    audio_bytes = base64.b64decode(full_audio_base64)
    
    # Padding fix for pcm16 byte alignment
    if len(audio_bytes) % 2 != 0:
        audio_bytes += b'\x00'
        
    audio_data = np.frombuffer(audio_bytes, dtype=np.int16)
    
    # 3. Return (Sample Rate, Data) to Gradio
    # OpenAI pcm16 is standard 24000Hz
    return (24000, audio_data)

                

In [22]:
def process_combined_chat(text_input, mic_audio):
    # Prepare the multimodal message content
    content = []
    
    # 1. Add text if it exists
    if text_input:
        content.append({"type": "text", "text": text_input})
    
    # 2. Add audio if it exists (Convert Gradio numpy to Base64 WAV)
    if mic_audio is not None:
        sr, y = mic_audio
        buffer = io.BytesIO()
        with wave.open(buffer, "wb") as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(sr)
            wav_file.writeframes(y.tobytes())
        base64_audio = base64.b64encode(buffer.getvalue()).decode("utf-8")
        content.append({"type": "input_audio", "input_audio": {"data": base64_audio, "format": "wav"}})

    if not content:
        return None # Nothing to send

    # 3. Call OpenRouter with the combined content
    stream = client.chat.completions.create(
        model=AUDIO_MODEL,
        modalities=["text", "audio"],
        audio={"voice": "alloy", "format": "pcm16"},
        stream=True,
        messages=[
            {
              "role": "user",
              "content": text_input   
            },
            {
                "role": "user",
                "content": [
                    {"type": "input_audio", "input_audio": {"data": base64_audio, "format": "wav"}}
                ]
            }
        ]
    )
     
    

    # 4. Stream response back to gr.Audio
    full_audio_base64 = ""
    for chunk in stream:
        if hasattr(chunk.choices[0].delta, 'audio') and chunk.choices[0].delta.audio:
            if 'data' in chunk.choices[0].delta.audio:
                # Append the raw base64 chunks
                full_audio_base64 += chunk.choices[0].delta.audio['data']

    if not full_audio_base64:
        return None

    # 2. Convert base64 to NumPy array (Gradio's preferred format)
    audio_bytes = base64.b64decode(full_audio_base64)
    
    # Padding fix for pcm16 byte alignment
    if len(audio_bytes) % 2 != 0:
        audio_bytes += b'\x00'
        
    audio_data = np.frombuffer(audio_bytes, dtype=np.int16)
    
    # 3. Return (Sample Rate, Data) to Gradio
    # OpenAI pcm16 is standard 24000Hz
    return text_input, (24000, audio_data)

In [20]:
voice = process_combined_chat("Hello, how are you?", None)

In [ ]:
# Handle chat messages
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = client.chat.completions.create(model=CHAT_MODEL, messages=messages)

    # while response.choices[0].finish_reason=="tool_calls":
    #     message = response.choices[0].message
    #     messages.append(message)
    #     response = client.chat.completions.create(model=CHAT_MODEL, messages=messages)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    audio = talker(reply)
    
    return history, audio

In [23]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        #image_output = gr.Image(height=500, interactive=False)
    
    with gr.Row():
        audio_output = gr.Audio(streaming=True, autoplay=True, label="AI Voice Response")

    with gr.Row():
        with gr.Column():
            message = gr.Textbox(label="Type your message", placeholder="Or use the mic below...")
            mic_in = gr.Audio(sources=["microphone"], type="numpy", label="Record Audio")
            submit_btn = gr.Button("Send audio")




    #mic_input.stop_recording(process_audio, inputs=message, outputs=audio_output)

    # Trigger function on button click with both inputs
    submit_btn.click(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(process_combined_chat, inputs=[chatbot, mic_in], outputs=[chatbot, audio_output])
    

    # submit_btn.click(process_audio, inputs=[mic_in], outputs=audio_output).then(
    #     chat, inputs=audio_output, outputs=audio_output
    # )

# Hooking up events to callbacks
    #input_text.submit(stream_audio_response, inputs=input_text, outputs=output_audio)
    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )

ui.launch(inbrowser=True, auth=("user", "pass"))

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


Opening in existing browser session.
